# Decode the Bioreactor — 09 Combined: best model per state
Each state is taken from the model that predicts it best:
- **`mech_ridge`** — our mechanistic model + per-day ridge correction (notebook 03): strong for **VCD / TCD**
- **`tobias_v3`** — Tobias' original v3 (his pipeline, branch `hybrid_v3_tr`): strong for **byproduct** (volume-balance / base sensor)
- **`blend`** — geometric mean of the two (often better than either when their errors differ)

Both models are evaluated **out-of-fold** on all 49 labelled batches (every prediction comes from a model that never saw that batch). The choice per state is made on the **dev** batches; the **hold-out** batches confirm it.
Note: Tobias' out-of-fold predictions come from his own 7 folds, ours from our 5 dev folds (+ hold-out predicted from the model fitted on dev) — both are honest per batch, but trained on slightly different numbers of batches.

## 0. Get Tobias' original v3 output (once)
Copies his result files from his branch into this folder, without switching branch:
```
git checkout hybrid_v3_tr -- outputs/hybrid_v3/oof_predictions.csv outputs/hybrid_v3/holdout_predictions.csv
```
`oof_predictions.csv` = his 7-fold out-of-fold predictions for the 49 batches, `holdout_predictions.csv` = his predictions for the 12 test batches (his leaderboard submission).

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from bioreactor.data import load, to_arrays, STATES
from bioreactor.model import NAMES, model_tag, predict_at_obs
from bioreactor.fit import fit_staged, initial_from_data
from bioreactor.hybrid import feature_table, ResidualCorrector

df, tmpl, train_ids, test_ids = load(ROOT / "data")
splits = pd.read_csv(ROOT / "outputs" / "splits.csv")
dev_ids = splits.query("role == 'dev'").batch_id.tolist()
hold_ids = splits.query("role == 'holdout'").batch_id.tolist()
fold_of = splits.query("role == 'dev'").set_index("batch_id").fold.to_dict()
role_of = splits.set_index("batch_id").role.to_dict()

# ---------------- settings
TOBIAS_V3_DIR = ROOT / "outputs" / "hybrid_v3"
TOBIAS_COL = "v3 final (50/50 blend)"        # which of his columns to use
AUTO_CHOICE = False                          # True: pick per state the best on the dev batches
CHOICE = {"vcd": "mech_ridge", "tcd": "mech_ridge", "substrate": "tobias_v3", "byproduct": "tobias_v3"}

MAX_NFEV_STAGE, MAX_NFEV, N_STARTS = 20, 60, 1      # only used if a mechanistic fit is not cached
FIT_KW = dict(residual="log", loss="soft_l1", f_scale=0.1)
CACHE = ROOT / "outputs" / "params_cache"
key = lambda t: np.round(np.asarray(t, float), 6)

## 1. Out-of-fold predictions of both models
**mech_ridge**: dev batch in fold f ← mechanistic fit + ridge on the other dev folds; hold-out batch ← fitted on all dev; test batch ← fitted on all 49. Mechanistic parameters come from the cache (`fold0..4`, `dev_all`, `all49`).

In [ ]:
def mech_params(ids, tag):
    f = CACHE / (f"{tag}.csv" if model_tag() == "v1" else f"{tag}_{model_tag()}.csv")
    if f.exists():
        s = pd.read_csv(f, index_col=0).iloc[:, 0]
        if list(s.index) == NAMES:
            print(f"[{tag}] from cache"); return s.values
    t0 = time.time()
    p, _ = fit_staged(to_arrays(df, ids), None, p_init=initial_from_data(df, ids), max_nfev_stage=MAX_NFEV_STAGE,
                      max_nfev_final=MAX_NFEV, n_starts=N_STARTS, verbose=False, **FIT_KW)
    pd.Series(p, index=NAMES).to_csv(f); print(f"[{tag}] fit {time.time()-t0:.0f} s")
    return p

def mech_ridge(tr_ids, pred_ids, tag, mask="measured"):
    """Fit mechanistic model (cached) + per-day ridge correction on tr_ids, predict pred_ids."""
    p = mech_params(tr_ids, tag)
    a_tr, a_p = to_arrays(df, tr_ids), to_arrays(df, pred_ids, sample_mask=mask)
    corr = ResidualCorrector("ridge_day").fit(feature_table(df, a_tr, predict_at_obs(p, a_tr)))
    F_p = feature_table(df, a_p, predict_at_obs(p, a_p))
    P = corr.predict(F_p)
    return pd.DataFrame({"batch_id": F_p.batch_id.values, "t_key": key(F_p.t), **{v: P[v].values for v in STATES}})

parts = []
for f in range(5):
    tr_ = [b for b in dev_ids if fold_of[b] != f]; va_ = [b for b in dev_ids if fold_of[b] == f]
    parts.append(mech_ridge(tr_, va_, f"fold{f}"))
parts.append(mech_ridge(dev_ids, hold_ids, "dev_all"))
parts.append(mech_ridge(train_ids, test_ids, "all49", mask="predict"))
PRED = {"mech_ridge": pd.concat(parts, ignore_index=True)}
print(PRED["mech_ridge"].batch_id.map(role_of).value_counts().to_dict())

In [ ]:
# Tobias' original v3: out-of-fold for the 49 labelled batches + his test predictions
oof_t = pd.read_csv(TOBIAS_V3_DIR / "oof_predictions.csv", dtype={"batch_id": str})
tob = pd.DataFrame({"batch_id": oof_t.batch_id, "t_key": key(oof_t.t),
                    **{v: oof_t[f"{TOBIAS_COL}|{v}"].values for v in STATES}})
hold_t = pd.read_csv(TOBIAS_V3_DIR / "holdout_predictions.csv", dtype={"batch_id": str})   # = our test batches
tob = pd.concat([tob, pd.DataFrame({"batch_id": hold_t.batch_id, "t_key": key(hold_t.elapsed_time),
                                    **{v: hold_t[v].values for v in STATES}})], ignore_index=True)
PRED["tobias_v3"] = tob
print(tob.batch_id.map(role_of).value_counts().to_dict())

In [ ]:
# observations + all candidates side by side; blend = geometric mean of the two models
obs = df[df.vcd.notna() & (df.elapsed_time > 0)]
OBS = pd.DataFrame({"batch_id": obs.batch_id.values, "t_key": key(obs.elapsed_time),
                    **{f"obs_{v}": obs[v].values for v in STATES}})
ALL = PRED["mech_ridge"].merge(PRED["tobias_v3"], on=["batch_id", "t_key"], how="outer",
                               suffixes=("@mech_ridge", "@tobias_v3"), validate="one_to_one")
for v in STATES:
    ALL[f"{v}@blend"] = np.sqrt(ALL[f"{v}@mech_ridge"] * ALL[f"{v}@tobias_v3"])
ALL = ALL.merge(OBS, on=["batch_id", "t_key"], how="left")
ALL["role"] = ALL.batch_id.map(role_of)
ALL["fold"] = ALL.batch_id.map(fold_of)        # dev folds 0-4 (NaN for hold-out / test)
ALL["day"] = np.rint(ALL.t_key).astype(int)
missing = ALL[[c for c in ALL.columns if "@" in c]].isna().sum()
assert not missing.any(), f"missing predictions (times not matched?): {missing[missing > 0].to_dict()}"
MODELS = ["mech_ridge", "tobias_v3", "blend"]
ALL.groupby("role").size()

## 2. MAPE per state and model — dev (choice) and hold-out (check)

In [ ]:
def mape(rows, model, v):
    return 100 * np.mean(np.abs(rows[f"{v}@{model}"] / rows[f"obs_{v}"] - 1))

tabs = {}
for role in ["dev", "holdout"]:
    R = ALL[ALL.role == role]
    tabs[role] = pd.DataFrame({m: {v: mape(R, m, v) for v in STATES} for m in MODELS}).T
    tabs[role]["score"] = tabs[role][STATES].mean(axis=1)
display(pd.concat(tabs, names=["batches", "model"]).round(2))

In [ ]:
if AUTO_CHOICE:
    CHOICE = {v: tabs["dev"][v].idxmin() for v in STATES}
print("choice per state:", CHOICE)

def combined(rows):
    P = pd.DataFrame({v: rows[f"{v}@{CHOICE[v]}"].values for v in STATES}, index=rows.index)
    P["tcd"] = np.maximum(P.tcd, P.vcd)          # total cells >= viable cells
    return P

res = {}
for role in ["dev", "holdout"]:
    R = ALL[ALL.role == role]; P = combined(R)
    res[role] = {v: 100 * np.mean(np.abs(P[v] / R[f"obs_{v}"] - 1)) for v in STATES}
    res[role]["score"] = np.mean([res[role][v] for v in STATES])
res = pd.DataFrame(res).T
display(pd.concat([tabs["dev"].assign(batches="dev"), tabs["holdout"].assign(batches="holdout"),
                   res.assign(batches=res.index).rename(index=lambda r: "COMBINED")])
        .reset_index().rename(columns={"index": "model"}).set_index(["batches", "model"]).round(2).sort_index())

## 3. Cross-validation of the combined model (5 dev folds)
Each dev fold's batches were predicted out-of-fold, so scoring them fold by fold gives the usual CV table (mean ± std over folds) for every single model, the blend and the combined model. Note: `CHOICE` (or `AUTO_CHOICE`) was made on all dev batches, so the combined row is slightly optimistic — the hold-out row in section 2 is the clean check.

In [ ]:
dev = ALL[ALL.role == "dev"]
cv_rows = []
for f in range(5):
    R = dev[dev.fold == f]
    for m in MODELS:
        s = {v: mape(R, m, v) for v in STATES}
        cv_rows.append({"model": m, "fold": f, **s, "score": np.mean(list(s.values()))})
    P = combined(R)
    s = {v: 100 * np.mean(np.abs(P[v] / R[f"obs_{v}"] - 1)) for v in STATES}
    cv_rows.append({"model": "COMBINED", "fold": f, **s, "score": np.mean(list(s.values()))})
cv = pd.DataFrame(cv_rows)
cv.to_csv(ROOT / "outputs" / "cv_scores_combined.csv", index=False)
display(cv.groupby("model")[STATES + ["score"]].agg(["mean", "std"]).round(2).sort_values(("score", "mean")))

In [ ]:
# score per fold: is the combined model better in every fold, or only on average?
piv = cv.pivot(index="fold", columns="model", values="score")
ax = piv.plot(kind="bar", figsize=(8, 3.5), rot=0)
ax.set_ylabel("challenge score (mean MAPE %)"); ax.set_title("score per CV fold"); plt.tight_layout(); plt.show()
print("folds where COMBINED is best:", int((piv.idxmin(axis=1) == "COMBINED").sum()), "of 5")

Error by day of the combined model vs. the two single models (dev batches, out-of-fold).

In [ ]:
R = ALL[ALL.role == "dev"]; P = combined(R)
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for a, v in zip(ax, STATES):
    for m in MODELS:
        e = 100 * np.abs(R[f"{v}@{m}"] / R[f"obs_{v}"] - 1)
        a.plot(e.groupby(R.day).mean(), "o--" if m != CHOICE[v] else "o-", ms=3, label=m)
    a.set_title(f"{v} (using {CHOICE[v]})"); a.set_xlabel("day"); a.set_ylabel("|% error|")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

## 4. Submission: combined predictions for the 12 test batches
mech_ridge test predictions come from the model fitted on all 49 batches; Tobias' are his submitted test predictions.

In [ ]:
RUN_FINAL = False
if RUN_FINAL:
    T = ALL[ALL.role == "test"].copy()
    P = combined(T)
    assert P.notna().all().all() and (P > 0).all().all()
    T[STATES] = P.values
    sub = tmpl.copy(); sub[STATES] = sub[STATES].astype(float)          # template columns are int zeros
    for b, g in sub.groupby("batch_id", sort=False):
        gt = T[T.batch_id == b].sort_values("t_key")
        assert len(g) == len(gt) and np.allclose(np.sort(g.elapsed_time.values), gt.t_key.values, atol=1e-3), b
        sub.loc[g.sort_values("elapsed_time").index, STATES] = gt[STATES].to_numpy()
    assert sub[STATES].notna().all().all()
    tag = "_".join(f"{v[:3]}-{CHOICE[v]}" for v in STATES)
    out = ROOT / "outputs" / f"submission_combined_{tag}.csv"
    sub.to_csv(out, index=False); print("written", out)
    display(sub.head())